# Advanced retrieval and evaluation

Read the [guide](README.md) for prerequisites, costs and exercises. Run all cells in order with the `notebooks`, `rag`, `memory` and `aws` groups installed. Calls use offline fixtures, not live model judgments.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_query_rewriting

[Script](examples/01_query_rewriting.py). Predict the result before running.


In [ ]:
"""Request bounded query rewrites, merge searches and deduplicate chunk IDs.

Run: uv run python modules/08_advanced_rag/examples/01_query_rewriting.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from pydantic import BaseModel, Field
from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.retrieval import lexical_search, load_chunks
from common.testing import tool_message


class Queries(BaseModel):
    queries: list[str] = Field(min_length=1, max_length=3)


def main() -> None:
    args, cfg = lesson_settings()
    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message(
                    "Queries",
                    {"queries": ["refund approval threshold", "refund approval expiration"]},
                )
            ],
        ),
        structured_output_model=Queries,
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent(
        "Rewrite: Who approves a refund and when does approval expire? "
        "Produce at most three short searches.",
        limits={"turns": 2},
    )
    if result.structured_output is None:
        raise ValueError("No queries within the turn budget")
    # KEY IDEA: cap both query count and per-query hits before merging evidence.
    hits = {}
    for query in result.structured_output.queries:
        for chunk in lexical_search(load_chunks(), query, k=2):
            hits[chunk.id] = chunk
    console.print("Unique evidence:", [c.record() for c in hits.values()])
    complete(result, cfg)


run_lesson(main, "modules/08_advanced_rag/examples/01_query_rewriting.py")

## 02_hybrid_search

[Script](examples/02_hybrid_search.py). Predict the result before running.


In [ ]:
"""Fuse BM25 and vector rankings without adding incomparable raw scores.

Run: uv run --group rag python modules/08_advanced_rag/examples/02_hybrid_search.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.lessons import complete, lesson_settings
from common.models import get_embeddings
from common.pretty import console, run_cli
from common.retrieval import hybrid_search, lexical_search, load_chunks


def main() -> None:
    args, cfg = lesson_settings()
    chunks = load_chunks()
    query = "full cancellation refund conditions"
    console.print("Lexical:", [c.id for c in lexical_search(chunks, query)])
    # KEY IDEA: reciprocal rank fusion combines ranks, then deduplicates by stable ID.
    console.print(
        "Hybrid:",
        [c.record() for c in hybrid_search(chunks, query, get_embeddings(mock=args.mock))],
    )
    complete()


run_lesson(main, "modules/08_advanced_rag/examples/02_hybrid_search.py")

## 03_reranking

[Script](examples/03_reranking.py). Predict the result before running.


In [ ]:
"""Rerank a candidate pool locally, then inspect the optional Bedrock contract.

Run: uv run python modules/08_advanced_rag/examples/03_reranking.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.aws_retrieval import bedrock_rerank
from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli
from common.retrieval import lexical_search, load_chunks, rerank


def main() -> None:
    args, cfg = lesson_settings()
    query = "approved refund timing"
    candidates = lexical_search(load_chunks(), query, k=5)
    # KEY IDEA: reranking cannot recover a document absent from the candidate pool.
    console.print("Local heuristic:", [c.id for c in rerank(query, candidates)])
    console.print(
        "Bedrock rerank:", bedrock_rerank(query, [c.text for c in candidates], mock=args.mock)
    )
    complete()


run_lesson(main, "modules/08_advanced_rag/examples/03_reranking.py")

## 04_filters_multihop

[Script](examples/04_filters_multihop.py). Predict the result before running.


In [ ]:
"""Bound a two-hop policy lookup and keep metadata filters explicit.

Run: uv run python modules/08_advanced_rag/examples/04_filters_multihop.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.retrieval import lexical_search, load_chunks
from common.testing import tool_message


def main() -> None:
    args, cfg = lesson_settings()
    chunks = load_chunks()
    # KEY IDEA: a second hop follows a policy reference, not an unbounded model loop.
    first = lexical_search(chunks, "full refund exception", k=2, topic="refund")
    next_query = (
        "full refund cancellation 24 hours 7 days"
        if any("POL-CAN-001" in c.text for c in first)
        else "cancellation"
    )
    second = lexical_search(chunks, next_query, k=2, topic="cancellation")
    console.print("Hop 1:", [c.record() for c in first])
    console.print("Hop 2:", [c.record() for c in second])
    calls = 0

    @tool
    def search_policy_topic(query: str, topic: str) -> list[dict]:
        """Search a bounded policy topic.

        Args:
            query: The question or policy cross-reference.
            topic: Either refund or cancellation for this lab.
        """
        nonlocal calls
        calls += 1
        if calls > 2 or topic not in {"refund", "cancellation"}:
            raise ValueError("Only two searches of approved topics are allowed")
        return [c.record() for c in lexical_search(chunks, query, k=2, topic=topic)]

    agent = Agent(
        model=get_model(
            cfg,
            responses=[
                tool_message(
                    "search_policy_topic", {"query": "full refund exception", "topic": "refund"}
                ),
                tool_message(
                    "search_policy_topic", {"query": next_query, "topic": "cancellation"}, "hop2"
                ),
                "Both the 24-hour booking window and the 7-day departure condition apply.",
            ],
        ),
        tools=[search_policy_topic],
        callback_handler=None,
        retry_strategy=None,
        system_prompt=(
            "Search refund policy, then follow its cancellation reference. "
            "Use at most two searches."
        ),
    )
    result = agent("What conditions determine a full refund?", limits={"turns": 3})
    complete(result, cfg)


run_lesson(main, "modules/08_advanced_rag/examples/04_filters_multihop.py")

## 05_evaluate_retrieval

[Script](examples/05_evaluate_retrieval.py). Predict the result before running.


In [ ]:
"""Measure held-out source hits and demonstrate a separate answer judge.

Run: uv run --group rag python modules/08_advanced_rag/examples/05_evaluate_retrieval.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.data import load_jsonl
from common.lessons import complete, lesson_settings
from common.models import get_embeddings
from common.pretty import console, run_cli
from common.retrieval_evaluation import judge_answer, retrieval_report


def main() -> None:
    args, cfg = lesson_settings()
    # KEY IDEA: a correct retrieved source does not establish a correct final answer.
    report = retrieval_report(load_jsonl("eval/policy_qa.jsonl"), get_embeddings(mock=args.mock))
    console.print({key: value for key, value in report.items() if key != "rows"})
    result = judge_answer(
        cfg,
        "When does approval expire?",
        "After 30 days.",
        "Approval expires after 30 minutes.",
        "30 minutes",
    )
    console.print("Separate judge:", result.structured_output)
    complete(result, cfg)


run_lesson(main, "modules/08_advanced_rag/examples/05_evaluate_retrieval.py")

## 06_failure_gallery

[Script](examples/06_failure_gallery.py). Predict the result before running.


In [ ]:
"""Expose false citations, missing evidence and untrusted retrieved instructions.

Run: uv run python modules/08_advanced_rag/examples/06_failure_gallery.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.data import data_path
from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli
from common.rag_agent import PolicyAnswer, validate_citations
from common.retrieval import load_chunks


def main() -> None:
    args, cfg = lesson_settings()
    chunk = load_chunks()[0]
    bad = PolicyAnswer(
        answer="Every trip is free.",
        abstained=False,
        citations=[
            {
                "chunk_id": chunk.id,
                "source": chunk.source,
                "page": chunk.page,
                "quote": "Every trip is free.",
            }
        ],
    )
    try:
        validate_citations(bad, [chunk])
    except ValueError as exc:
        console.print("Rejected false citation:", str(exc))
    # KEY IDEA: quarantine poison; quote validation alone cannot stop injection.
    poisoned = data_path("security/poisoned_doc.md").read_text(encoding="utf-8")
    console.print("Untrusted document excluded from approved corpus:", len(poisoned), "characters")
    safe = PolicyAnswer(answer="I do not know from these sources.", abstained=True, citations=[])
    validate_citations(safe, [])
    console.print("Missing evidence:", safe.answer)
    complete()


run_lesson(main, "modules/08_advanced_rag/examples/06_failure_gallery.py")

## retrieval_comparison

[Script](use_case/retrieval_comparison.py). Predict the result before running.


In [ ]:
"""Export measured before/after retrieval results without claiming invented improvements.

Run: uv run --group rag python modules/08_advanced_rag/use_case/retrieval_comparison.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import argparse
import json

from common.config import ROOT
from common.data import load_jsonl
from common.lessons import complete, lesson_settings
from common.models import get_embeddings
from common.pretty import console, run_cli
from common.retrieval_evaluation import evaluate_answers, retrieval_report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--judge-answers", action="store_true", help="Generate and judge all 12 answers"
    )
    args, cfg = lesson_settings(parser)
    # KEY IDEA: report the observed difference, including a tie or regression.
    report = retrieval_report(load_jsonl("eval/policy_qa.jsonl"), get_embeddings(mock=args.mock))
    if args.judge_answers:
        report["answer_evaluation"] = evaluate_answers(cfg, load_jsonl("eval/policy_qa.jsonl"))
    path = ROOT / ".nimbus/retrieval_report.json"
    path.parent.mkdir(exist_ok=True)
    path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    console.print("Method | source hit rate @3")
    console.print("Lexical |", report["baseline_hit_rate"])
    console.print("Hybrid  |", report["hybrid_hit_rate"])
    console.print("Report:", str(path), "Mode:", report["mode"])
    complete()


run_lesson(main, "modules/08_advanced_rag/use_case/retrieval_comparison.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
